In [50]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [51]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
model_path = os.getenv("RF_MODEL_PATH")
selected_features_path = os.getenv("SELECTED_FEATURES_PATH")
hyperparameters_path = os.getenv("HYPERPARAMETERS_PATH")
pipeline_path = os.getenv("PIPELINE_PATH")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"model_path: {model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"hyperparameters_path: {hyperparameters_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

# Hardcoded variables
VARIANCE_THRESHOLD=0.01  # REMOVE NEAR-CONSTANT GENES
FEATURE_SELECTION_PERCENTILE=20  # KEEP TOP n% OF FEATURES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
model_path: /home/mariya/Data/pallas/training/gex/gex.model.pkl
selected_features_path: /home/mariya/Data/pallas/training/gex/gex.selected_features.pkl
hyperparameters_path: /home/mariya/Data/pallas/training/gex/gex.hyperparameters.pkl
pipeline_path: /home/mariya/Data/pallas/training/gex/gex.pipeline.pkl
random_seed: 42
n_jobs: 10


In [52]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
817_B,-1.171806,-1.426614,4.694081,5.052904,3.602915,7.305013,-1.346290,4.879798,4.628552,6.154961,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
817_T,2.572748,-1.426614,5.128790,5.109519,3.024993,5.821290,3.740545,4.011223,4.737872,5.939192,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
845_T,2.658142,-1.426614,5.317063,5.266773,3.237485,5.120946,3.478314,4.342898,4.782189,5.946979,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
858_B,-1.216338,-1.426614,5.023544,5.241379,3.511032,6.696712,-0.786851,4.345535,5.134837,6.212021,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
858_T,2.587271,-1.426614,5.083224,5.390539,3.158276,4.899801,4.423113,3.602772,4.853483,6.658247,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJPHALL020044,-1.426614,-1.426614,3.237993,4.077183,4.232792,7.664968,3.929772,5.095714,4.711884,6.264560,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
SJPHALL020046,-1.426614,-1.426614,3.002809,4.478303,4.056364,5.656712,5.884539,3.027721,4.302519,6.892578,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
SJPHALL021,-1.426614,-1.426614,3.564498,4.191432,3.798929,6.735999,3.864376,5.606709,5.048537,6.486354,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614


In [53]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
AML015-D,"AML, KMT2A-r"
AML092-D,"AML, CBFB::MYH11"
AML097-D,"AML, RUNX1::RUNX1T1"
AML054-D,"AML, NPM1alt"
AML095-D,"AML, NPM1alt"
...,...
SJHBBBM4PX3,"B-ALL, high hyperdiploid"
SJHBBBM4X52,"B-ALL, ETV6::RUNX1"
SJHBBBM5GW9,"B-ALL, high hyperdiploid"


In [54]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
AML015-D,-1.123126,-1.426614,4.051135,4.835182,4.364201,5.917704,-1.153784,6.604805,4.983698,5.590614,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
AML092-D,-1.426614,-1.426614,4.671866,3.419581,3.215257,8.278960,-0.758350,7.047270,4.778793,4.982002,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
AML097-D,-0.759139,-1.426614,3.995396,4.188398,4.844314,5.148712,1.329120,6.156535,4.925816,6.421458,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
AML054-D,2.580283,-1.426614,4.773521,3.293920,5.014208,3.457228,1.634329,6.858053,7.588965,5.844989,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
AML095-D,0.233009,-1.426614,4.506679,3.574045,3.303550,4.668761,5.737484,6.734536,5.093864,5.203428,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBM4PX3,-1.294903,-1.426614,4.173994,4.110167,4.854004,3.446063,6.032661,0.424837,4.412007,6.156965,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
SJHBBBM4X52,-1.035433,-1.426614,3.718090,4.181087,4.198508,6.548175,4.627612,2.821258,4.295883,6.613602,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614
SJHBBBM5GW9,-0.923373,-1.426614,3.767298,4.200982,3.901331,6.404409,5.816699,2.375962,2.206681,6.742112,...,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614,-1.426614


In [55]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
SJHYPER001,"B-ALL, high hyperdiploid"
SJBALL016242,"B-ALL, ETV6::RUNX1-like"
SJBALL046123,"B-ALL, PAX5 P80R"
SJBALL013909,"B-ALL, KMT2A-r"
SJALL050001,"B-ALL, KMT2A-r"
...,...
AML051-D,"AML, NPM1alt"
AML101-D,"AML, CBFB::MYH11"
AML011-D,"AML, RUNX1::RUNX1T1"


In [56]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJHYPER001,-0.722346,-1.282907,3.990706,4.520484,4.396938,5.747752,-1.184799,3.507780,3.669045,6.353644,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907
SJBALL016242,-0.989296,-1.282907,4.409275,3.586161,4.040752,4.501467,-0.099562,2.329639,4.308617,5.829121,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907
SJBALL046123,0.923524,-0.786235,4.414047,3.343875,4.244156,3.163674,4.764234,4.616258,5.772816,6.130686,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907
SJBALL013909,-0.277160,-1.282907,3.975505,2.887586,4.353482,3.049614,1.799260,5.396522,4.127169,6.512534,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907
SJALL050001,-0.276975,-1.282907,4.841612,4.715474,4.102610,5.895272,-0.477964,4.841939,4.809185,6.168048,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AML051-D,-1.282907,-1.282907,3.648659,4.653945,4.156091,5.591664,0.612529,3.909912,4.726638,6.969134,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907
AML101-D,-1.282907,-1.282907,4.487046,2.435455,2.902080,9.122686,-0.943737,4.907101,4.227243,4.577170,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907
AML011-D,-1.282907,-1.282907,4.273998,4.047313,4.602813,3.376474,-0.309857,4.443786,4.814288,5.845824,...,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907,-1.282907


In [57]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(5)

In [58]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, PAX5 P80R', 'B-ALL, KMT2A-r', 'B-ALL, KMT2A-r',
       'B-ALL, DUX4-r', 'B-ALL, KMT2A-r', 'B-ALL, ETV6::RUNX1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, DUX4-r', 'B-ALL, BCR::ABL1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', 'B-ALL, DUX4-r',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r', 'B-ALL, ZNF384-r',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, high hyperdiploid', 'B-ALL, ZNF384-r', 'B-ALL, BCR::ABL1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, high hyperdiploid',
       'B-ALL, BCR::ABL1-like', 'B-ALL, hypodiploid', 'B-ALL, NUTM1-r',
       'B-ALL, KMT2A-r', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, BCR::ABL1',
       'B-ALL, high hyperdiploid', 'B-ALL, TCF3::PBX1',
   

In [64]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [5, 10, 20],
    'rf__min_samples_leaf': [2, 5, 10],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparameters_path)

# Save features
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, selected_features_path)

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


Best hyperparameters: {'rf__n_estimators': 50, 'rf__min_samples_split': 20, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9126064971499094


['/home/mariya/Data/pallas/training/gex/gex.pipeline.pkl']

In [65]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparameters_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, model_path)

Final model parameters: {'n_estimators': 500, 'min_samples_split': 20, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  40 tasks      | elapsed:  7.9min
[Parallel(n_jobs=-10)]: Done  50 out of  50 | elapsed:  9.8min finished


Cross-validation F1-weighted: 0.920 ± 0.010


['/home/mariya/Data/pallas/training/gex/gex.model.pkl']

In [66]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3923
Shape of X_test_selected: (753, 3923)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         1
     AML, CBFA2T3::GLIS2      1.000     1.000     1.000         7
        AML, CBFB::MYH11      1.000     0.935     0.967        31
           AML, CEBPAalt      1.000     1.000     1.000        13
        AML, DEK::NUP214      1.000     0.167     0.286         6
             AML, ETV6-r      0.000     0.000     0.000         2
           AML, FUS::ERG      1.000     1.000     1.000         1
            AML, KAT6A-r      1.000     1.000     1.000         2
            AML, KMT2A-r      0.839     0.783     0.810        60
            AML, MECOM-r      1.000     1.000     1.000         1
            AML, NPM1alt      0.720     0.818     0.766        22
            AML, NUP98-r      0.889     0.500     0.640        16
        AML,

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [62]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
          Gene  Importance
3148     NUTM1    0.004246
1826    MEGF10    0.003838
337    RUNX1T1    0.003719
3716     APELA    0.003496
2131     AGAP1    0.002940
3192      PBX1    0.002862
3162    ANKS1B    0.002766
538      SALL4    0.002628
789        HLF    0.002391
2180   IGF2BP1    0.002381
3772  SLC22A31    0.002164
1766     MEIS1    0.002131
156    ARHGAP6    0.002060
3331     GREB1    0.002048
1964   SLC25A4    0.002026
334      HOXA9    0.002017
3026    SKIDA1    0.002014
3083     HOXB4    0.001948
1036     HOXB3    0.001947
773      HOXB6    0.001944


In [63]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 1 out of 3923

Bottom 20 genes by importance (final model):
          Gene    Importance
1581     PPM1B  4.063950e-06
3764       NaN  3.975687e-06
3264     FANCA  3.967246e-06
3595     TREX1  3.918832e-06
3863   ZNF280B  3.868544e-06
1127     DNPEP  3.736862e-06
3793     IKBKE  3.645064e-06
3918      F8A1  3.593253e-06
1112  SLC25A16  3.583212e-06
3917    PRRC2B  3.163828e-06
2533     ZNF83  2.925634e-06
3830       NaN  2.915566e-06
3671  ZNF593OS  2.855861e-06
3814       NaN  2.846523e-06
3905      DUXB  2.639760e-06
3853     CWC25  2.385797e-06
1329     MGAT1  1.538213e-06
2806      NET1  1.388072e-06
3640  DEFB131B  2.293907e-20
3782      BOP1  0.000000e+00
